# Hyperparameter Tuning using GridSearchCV

## Objective

The objective of this notebook is to improve the performance of a Machine Learning model by finding the best combination of hyperparameters using GridSearchCV.

In this notebook, you will learn how to:

- Load the dataset
- Handle missing values
- Define features and target variable
- Create a Machine Learning pipeline
- Define a hyperparameter grid
- Perform Grid Search with Cross Validation
- Display the best hyperparameters
- Evaluate the best model

In [1]:
# Import Required Libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

**Step 1: Mount Google Drive**

Google Drive is mounted to access the dataset and save the trained model.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


**Step 2: Load the Dataset**

Load the Employee Salary Classification dataset from Google Drive.

In [3]:
df = pd.read_excel(
    "/content/drive/MyDrive/Datasets/Salary_Logistic_Regression_Multiclass_Classification_Dataset.xlsx"
)

df

,Experience_Years,Education_Level,Age,Monthly_Salary_PKR,Salary_Category
0,1,12.0,22.0,40000,0
1,2,12.0,23.0,45000,0
2,2,14.0,24.0,50000,0
3,3,14.0,25.0,58000,0
4,3,NaN,26.0,65000,0
5,4,16.0,27.0,70000,0
6,4,16.0,28.0,74000,0
7,5,16.0,29.0,82000,1
8,5,18.0,30.0,90000,1
9,6,18.0,31.0,98000,1


## Step 3: Handle Missing Values

Check whether the dataset contains missing values.

If any missing values are found, replace them with the median value of the respective column.

In [4]:
print("\nBefore Filling Missing Values")
print(df.isnull().sum())

df["Education_Level"] = df["Education_Level"].fillna(
    df["Education_Level"].median()
)

df["Age"] = df["Age"].fillna(
    df["Age"].median()
)

print("\nAfter Filling Missing Values")

print(df.isnull().sum())


Before Filling Missing Values
Experience_Years      0
Education_Level       2
Age                   1
Monthly_Salary_PKR    0
Salary_Category       0
dtype: int64

After Filling Missing Values
Experience_Years      0
Education_Level       0
Age                   0
Monthly_Salary_PKR    0
Salary_Category       0
dtype: int64


## Step 4: Define Features and Target Variable

The independent variables are:

- Experience Years
- Education Level
- Age

The dependent variable is:

- Salary Category

Salary Categories:

- 0 = Low Salary
- 1 = Medium Salary
- 2 = High Salary
- 3 = Executive Salary

In [5]:
X = df[
    [
        "Experience_Years",
        "Education_Level",
        "Age"
    ]
]

y = df["Salary_Category"]

## Step 5: Split the Dataset

Split the dataset into training and testing sets.

- 80% Training Data
- 20% Testing Data

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Step 6: Create a Machine Learning Pipeline

Create a pipeline that first scales the features using **StandardScaler** and then trains a **Random Forest Classification** model.

Using a pipeline ensures that preprocessing is applied correctly during Grid Search.

In [8]:
pipeline = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=42
            )
        )
    ]
)

## Step 7: Define the Hyperparameter Grid

Create a grid containing different combinations of hyperparameter values.

GridSearchCV will test every combination and identify the one that produces the best model performance.

In [9]:
param_grid = {

    "model__n_estimators": [
        50,
        100,
        150
    ],

    "model__max_depth": [
        3,
        5,
        7,
        None
    ],

    "model__min_samples_split": [
        2,
        5,
        10
    ]
}

## Step 8: Perform Hyperparameter Tuning

Use **GridSearchCV** to train the model using every combination of hyperparameters.

The best combination will be selected based on the highest Cross Validation accuracy.

In [11]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

print("Grid Search Completed Successfully")

Grid Search Completed Successfully


## Step 9: Display the Best Hyperparameters

Display the best combination of hyperparameters found by GridSearchCV.

Also display the best Cross Validation accuracy achieved during the search.

In [12]:
print("Best Hyperparameters:")

print(grid_search.best_params_)

print()

print("Best Cross Validation Accuracy:")

print(
    round(grid_search.best_score_ * 100,2),
    "%"
)

Best Hyperparameters:
{'model__max_depth': 3, 'model__min_samples_split': 2, 'model__n_estimators': 100}

Best Cross Validation Accuracy:
97.5 %


## Step 10: Make Predictions Using the Best Model

Use the best model identified by GridSearchCV to predict the salary categories for the testing dataset.

In [13]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(
    X_test
)

print(y_pred)

[1 1 3 1 2 2 3 2 3 2]


## Step 11: Evaluate Model Performance

Evaluate the best model using:

- Accuracy
- Confusion Matrix
- Classification Report

In [14]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:")

print(
    round(accuracy * 100,2),
    "%"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy:
90.0 %

Confusion Matrix:
[[3 0 0]
 [0 4 1]
 [0 0 2]]

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
           2       1.00      0.80      0.89         5
           3       0.67      1.00      0.80         2

    accuracy                           0.90        10
   macro avg       0.89      0.93      0.90        10
weighted avg       0.93      0.90      0.90        10



## Step 12: Predict the Salary Category for a New Employee

Predict the salary category for an employee with:

- Experience = 12 Years
- Education Level = 20 Years
- Age = 40 Years

In [15]:
prediction = best_model.predict(
    [[12,20,40]]
)

salary_categories = {
    0: "Low Salary",
    1: "Medium Salary",
    2: "High Salary",
    3: "Executive Salary"
}

print("Predicted Salary Category:")

print(
    salary_categories[prediction[0]]
)

Predicted Salary Category:
High Salary


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


# Conclusion

In this notebook, GridSearchCV was used to automatically search for the best combination of hyperparameters for a Random Forest Classification model. A Machine Learning pipeline combined feature scaling and model training into a single workflow. The best model was selected based on Cross Validation accuracy and was then evaluated using accuracy, a confusion matrix, and a classification report. Finally, the optimized model was used to predict the salary category of a new employee.